In [2]:
!pip install pyproj
!pip install geopandas


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ============================================================
# So2Sat LCZ42 – Geolocation pipeline (validated corrections)
# ============================================================
#
# Corrections applied:
#  - EPSG 327xx false northing fix (y < 0 → +10,000,000)
#  - London: EPSG +1
#  - China cities:
#       * Nanjing    → EPSG -1
#       * Hong Kong  → EPSG -1
#       * Shenzhen   → EPSG +1
#
# Explicitly NOT corrected:
#  - Cuba cluster (LCZ 7): dataset-level error
#
# Output:
#  - GeoPackage with corrected lon/lat, LCZ class, EPSG info, split
#
# ============================================================

import os
import sys
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "cromacode"))
from geo_utils import epsg327_false_northing_fix, to_lonlat, points_gdf_from_lonlat

# ----------------------------
# Paths
# ----------------------------
H5_IMG = "D:/LCZ42/per_continent/testing_autre.h5"
H5_GEO = "D:/LCZ42/per_continent/testing_geo_autre.h5"


H5_DIR_CONTINENTS = "D:/LCZ42/per_continent/"

CONTINENT_TO_H5 = {
    "europe": H5_DIR_CONTINENTS + "testing_europe.h5",
    "afrique": H5_DIR_CONTINENTS + "testing_afrique.h5",
    "asie_ouest": H5_DIR_CONTINENTS + "testing_asie_ouest.h5",
    "asie_est": H5_DIR_CONTINENTS + "testing_asie_est.h5",
    "ameriquenord": H5_DIR_CONTINENTS + "testing_ameriquenord.h5",
    "ameriquesud": H5_DIR_CONTINENTS + "testing_ameriquesud.h5",
    "oceanie": H5_DIR_CONTINENTS + "testing_oceanie.h5",
    "autre": H5_DIR_CONTINENTS + "testing_autre.h5",
}

# CSV files exported from QGIS selections (must contain column "idx")
CSV_LONDON = "./idx_london.csv"        # EPSG +1
CSV_NANJING = "./idx_nanjing.csv"      # EPSG -1
CSV_HONGKONG = "./idx_hongkong.csv"    # EPSG -1
CSV_SHENZHEN = "./idx_shenzhen.csv"    # EPSG +1

OUT_GPKG = "./so2sat_lcz42_testing_points_continent_na.gpkg"

# ----------------------------
# Helper functions
# ----------------------------
def load_idx_csv(csv_path: str) -> np.ndarray:
    """Load unique idx values from a QGIS-exported CSV."""
    df = pd.read_csv(csv_path)
    if "idx" not in df.columns:
        raise ValueError('CSV must contain a column named "idx".')
    idx = (
        df["idx"]
        .astype(str)
        .str.replace('"', "", regex=False)
        .astype(np.int64)
        .unique()
    )
    return idx


# ----------------------------
# Load LCZ labels (one-hot → class) + split
# ----------------------------
with h5py.File(H5_IMG, "r") as f:
    labels_onehot = f["label"][:]        # (N, 17)
    #split = f["split"][:].astype(str)    # (N,)

lcz = np.argmax(labels_onehot, axis=1).astype(np.int32) + 1

# ----------------------------
# Load geo data
# ----------------------------
with h5py.File(H5_GEO, "r") as f:
    coord = f["coord"][:]                # (N, 6)
    epsg_orig = f["epsg"][:, 0]          # (N,)

x = coord[:, 0].astype(np.float64)
y = coord[:, 1].astype(np.float64)
epsg_orig = epsg_orig.astype(np.int64)

# ----------------------------
# Apply standard EPSG 327xx fix
# ----------------------------
y_fixed = epsg327_false_northing_fix(y, epsg_orig)

# ----------------------------
# Apply validated EPSG corrections (by idx lists)
# ----------------------------
epsg_fixed = epsg_orig.copy()

fix_specs = [
    (CSV_LONDON, +1),
    (CSV_NANJING, -1),
    (CSV_HONGKONG, -1),
    (CSV_SHENZHEN, +1),
]

for csv_path, offset in fix_specs:
    if csv_path is None or not os.path.exists(csv_path):
        continue
    idx_list = load_idx_csv(csv_path)
    epsg_fixed[idx_list] = epsg_fixed[idx_list] + int(offset)

print("Applied EPSG fixes (by idx lists).")

# ----------------------------
# Final lon/lat after corrections
# ----------------------------
lon, lat = to_lonlat(x, y_fixed, epsg_fixed)

# ----------------------------
# Build GeoDataFrame
# ----------------------------
gdf = points_gdf_from_lonlat(
    lon, lat,
    extra_cols={
        #"split": split,
        "epsg_orig": epsg_orig,
        "epsg_fixed": epsg_fixed,
        "lcz": lcz,
    },
)

# ----------------------------
# Export GeoPackage
# ----------------------------
if os.path.exists(OUT_GPKG):
    os.remove(OUT_GPKG)

gdf.to_file(OUT_GPKG, driver="GPKG")
print(f"\nSaved {len(gdf)} points to:\n  {OUT_GPKG}")


In [ ]:
# ============================================================
# Paths
# ============================================================
POINTS_GPKG = "./so2sat_lcz42_points.gpkg"
# Je crois que le SHP sert à ajouter des informations importantes à notre fichier gpkg et 
# surtout de garder que les données sur les pays avec les 'polygones pays'
from geo_utils import COUNTRIES_SHP

# ============================================================
# Load data
# ============================================================
pts = gpd.read_file(POINTS_GPKG)


countries = gpd.read_file(COUNTRIES_SHP).to_crs("EPSG:4326")

# Keep useful fields (robust)
keep_cols = []
for c in ["ADMIN", "CONTINENT", "NAME", "NAME_EN", "SOVEREIGNT"]:
    if c in countries.columns:
        keep_cols.append(c)

countries = countries[keep_cols + ["geometry"]].copy()

# ============================================================
# Spatial join
# ============================================================
j = gpd.sjoin(pts, countries, predicate="within", how="left")

# Harmonize names
if "ADMIN" in j.columns:
    j["country"] = j["ADMIN"]
elif "NAME_EN" in j.columns:
    j["country"] = j["NAME_EN"]
elif "NAME" in j.columns:
    j["country"] = j["NAME"]
else:
    j["country"] = np.nan

j["continent"] = j["CONTINENT"] if "CONTINENT" in j.columns else np.nan
j["is_land"] = ~j["country"].isna()

# ============================================================
# Global summary
# ============================================================
n_total = len(j)
n_land = int(j["is_land"].sum())
n_ocean = n_total - n_land

print("\n=== Global counts ===")
print(f"Total tiles      : {n_total}")
print(f"Land tiles       : {n_land}")
print(f"Ocean tiles      : {n_ocean}")
print(f"Ocean fraction   : {n_ocean / n_total:.3%}")

# ============================================================
# Tiles per continent + per country (hierarchical)
# ============================================================
print("\n=== Tiles per continent >> country ===")

tiles_continent = (
    j[j["is_land"]]
    .groupby("continent")
    .size()
    .sort_values(ascending=False)
)

tiles_country = (
    j[j["is_land"]]
    .groupby(["continent", "country"])
    .size()
    .sort_values(ascending=False)
)

for continent, n_cont in tiles_continent.items():
    print(f"\n{continent} ({n_cont})")

    sub = tiles_country.loc[continent]
    for country, n_country in sub.items():
        print(f"      {country}: {n_country}")

# ============================================================
# LCZ distribution per continent (percentages)
# ============================================================
print("\n=== LCZ class distribution per continent (%) ===")

lcz_counts = (
    j[j["is_land"]]
    .groupby(["continent", "lcz"])
    .size()
    .reset_index(name="count")
)

# Cas monde entier
world_counts = (
    j[j["is_land"]]
    .groupby("lcz")
    .size()
    .reset_index(name="count")
)
world_total = world_counts["count"].sum()
print(f"\nWorld")
for _, row in world_counts.sort_values("lcz").iterrows():
    pct = 100.0 * row["count"] / world_total
    print(f"  LCZ {int(row['lcz']):2d}: {pct:6.2f} %")

# Cas par continent
for continent, dfc in lcz_counts.groupby("continent"):
    total = dfc["count"].sum()

    print(f"\n{continent}")
    dfc = dfc.sort_values("lcz")
    for _, row in dfc.iterrows():
        pct = 100.0 * row["count"] / total
        print(f"  LCZ {int(row['lcz']):2d}: {pct:6.2f} %")


In [3]:
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

# ============================================================
# Paths
# ============================================================
POINTS_GPKG = "./so2sat_lcz42_points.gpkg"
# Classification climatique mondiale 
KOPPEN_TIF = "D:/koppen_geiger_tif/1991_2020/koppen_geiger_0p1.tif"

# ============================================================
# Load points
# ============================================================
print("Loading LCZ points...")
pts = gpd.read_file(POINTS_GPKG)

# We assume points are in EPSG:4326
if pts.crs is None:
    raise ValueError("Points GeoPackage has no CRS defined")

# ============================================================
# Sample Köppen raster at point locations
# ============================================================
print("Sampling Köppen raster at point locations...")

with rasterio.open(KOPPEN_TIF) as src:
    # Reproject raster on the fly if needed
    if src.crs != pts.crs:
        raster = WarpedVRT(
            src,
            crs=pts.crs,
            resampling=Resampling.nearest,
        )
    else:
        raster = src

    coords = [(geom.x, geom.y) for geom in pts.geometry]
    koppen_vals = np.array(
        [v[0] for v in raster.sample(coords)],
        dtype=np.float32,
    )

    nodata = raster.nodata

# Handle NoData
koppen_vals = np.where(koppen_vals == nodata, np.nan, koppen_vals)
pts["koppen"] = koppen_vals

# ============================================================
# Filter: land only (optional but recommended)
# ------------------------------------------------------------
# We rely on geometry validity: points with NaN koppen are excluded
# ============================================================
df = pts[pts["koppen"].notna()].copy()

# ============================================================
# Köppen × LCZ (counts)
# ============================================================
print("\n=== Köppen × LCZ (counts) ===")

koppen_lcz_counts = (
    df.groupby(["koppen", "lcz"])
    .size()
    .reset_index(name="n_tiles")
    .sort_values(["koppen", "lcz"])
)

print(koppen_lcz_counts.to_string(index=False))

# ============================================================
# Köppen × LCZ (percentages per Köppen class)
# ============================================================
print("\n=== Köppen × LCZ (percentages per Köppen class) ===")

koppen_totals = (
    koppen_lcz_counts
    .groupby("koppen")["n_tiles"]
    .sum()
    .rename("total")
)

koppen_lcz_pct = koppen_lcz_counts.merge(
    koppen_totals,
    on="koppen",
)

koppen_lcz_pct["percent"] = (
    100.0 * koppen_lcz_pct["n_tiles"] / koppen_lcz_pct["total"]
)

for koppen, dfk in koppen_lcz_pct.groupby("koppen"):
    print(f"\nKöppen {int(koppen)}")
    for _, r in dfk.sort_values("lcz").iterrows():
        print(
            f"  LCZ {int(r['lcz']):2d}: "
            f"{r['percent']:6.2f} % "
            f"({int(r['n_tiles'])})"
        )


Loading LCZ points...
Sampling Köppen raster at point locations...

=== Köppen × LCZ (counts) ===
 koppen  lcz  n_tiles
    1.0    1       57
    1.0    2        8
    1.0    3     1292
    1.0    4       13
    1.0    5       12
    1.0    6      118
    1.0    7      119
    1.0    8     1017
    1.0    9      338
    1.0   10      118
    1.0   11      471
    1.0   12       52
    1.0   14       90
    1.0   15        7
    1.0   17      362
    2.0    1       68
    2.0    2      437
    2.0    3      180
    2.0    4      283
    2.0    5      332
    2.0    6       25
    2.0    7      701
    2.0    8      527
    2.0    9       78
    2.0   10      297
    2.0   11     1176
    2.0   12       27
    2.0   13       48
    2.0   14     1030
    2.0   15       54
    2.0   16       52
    2.0   17      366
    3.0    1       84
    3.0    2      359
    3.0    3     2151
    3.0    4      282
    3.0    5       71
    3.0    6     1248
    3.0    7      119
    3.0    8      720
